# Anexo · Imágenes: genera con Stable Diffusion y deja que la máquina las mire
*Curso «Google Colab» · Anexo multimodal · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** generar imágenes con **Stable Diffusion** en la GPU gratis y después pedirle a otros modelos que las
**clasifiquen**, las **describan** y las **comparen con frases**.

| Modelo | Para qué | Licencia |
|---|---|---|
| `stable-diffusion-v1-5/stable-diffusion-v1-5` | Texto → imagen | CreativeML OpenRAIL-M (con restricciones de uso) |
| `google/vit-base-patch16-224` | Clasificar en 1.000 categorías | Apache 2.0 |
| `Salesforce/blip-image-captioning-base` | Describir una imagen | BSD-3 |
| `openai/clip-vit-base-patch32` | Comparar una imagen con frases | MIT (repositorio de OpenAI) |

> **Uso responsable:** la licencia OpenRAIL-M prohíbe, entre otros, usar el modelo para dañar, engañar o suplantar a
> personas, y Colab prohíbe crear *deepfakes*. Genera paisajes, objetos, ilustraciones: no personas reales.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os, time
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
TIPO = torch.float16 if DISPOSITIVO == "cuda" else torch.float32
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

In [ ]:
%pip install -q diffusers transformers accelerate

## 1. Generar con Stable Diffusion

La difusión parte de **ruido** y lo va limpiando, paso a paso, hacia una imagen que calce con tu texto.

- `num_inference_steps`: cuántos pasos de limpieza (más pasos, más detalle y más tiempo).
- `guidance_scale`: cuánto obedece al texto (7 a 8 es lo usual).
- `negative_prompt`: lo que **no** quieres en la imagen.
- La **semilla** fija el ruido inicial: misma semilla y mismo texto, misma imagen.

In [ ]:
from diffusers import StableDiffusionPipeline

pintor = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=TIPO).to(DISPOSITIVO)

def pintar(texto, semilla=0, pasos=30, guia=7.5, negativo="borroso, deforme, texto, marca de agua"):
    generador = torch.Generator(DISPOSITIVO).manual_seed(semilla)
    inicio = time.perf_counter()
    imagen = pintor(texto, negative_prompt=negativo, num_inference_steps=pasos, guidance_scale=guia, generator=generador).images[0]
    print(f"{pasos} pasos en {time.perf_counter() - inicio:.1f} s")
    return imagen

# SD 1.5 entiende mejor el inglés
textos = ["a red mountain bike leaning on a wooden fence, photo", "a cup of coffee on a desk next to a laptop, photo",
          "a lighthouse on a rocky coast at sunset, painting"]
imagenes = [pintar(t, semilla=i) for i, t in enumerate(textos)]
from IPython.display import display
for imagen in imagenes:
    display(imagen.resize((256, 256)))

> **Experimenta:** la misma frase con `pasos=5`, `15` y `50`; y `guia=2` frente a `guia=12`.

## 2. Clasificar con ViT (1.000 categorías de ImageNet)

In [ ]:
from transformers import pipeline
clasificador = pipeline("image-classification", model="google/vit-base-patch16-224", device=DISPOSITIVO)
for texto, imagen in zip(textos, imagenes):
    mejores = clasificador(imagen, top_k=3)
    print(f"«{texto[:40]}…» →", ", ".join(f"{r['label']} ({r['score']:.0%})" for r in mejores))

## 3. Describir con BLIP

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base", torch_dtype=TIPO).to(DISPOSITIVO)
for imagen in imagenes:
    entrada = blip_proc(images=imagen, return_tensors="pt").to(DISPOSITIVO, TIPO)
    print("BLIP dice:", blip_proc.decode(blip.generate(**entrada, max_new_tokens=30)[0], skip_special_tokens=True))

## 4. Comparar con frases: CLIP (clasificar sin entrenar)

CLIP pone imágenes y textos en el mismo espacio: puede decir qué frase calza mejor con una imagen, aunque esas
categorías nunca las haya visto como tales (*zero-shot*).

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DISPOSITIVO)
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
frases = ["a bicycle", "a cup of coffee", "a lighthouse", "a cat", "a city at night"]
entrada = clip_proc(text=frases, images=imagenes, return_tensors="pt", padding=True).to(DISPOSITIVO)
with torch.no_grad():
    probabilidades = clip(**entrada).logits_per_image.softmax(dim=1).cpu()
for texto, fila in zip(textos, probabilidades):
    print(f"«{texto[:35]}…» → {frases[int(fila.argmax())]} ({float(fila.max()):.0%})")

## 5. Tu propia foto (opcional): cámara o archivo

En Colab puedes tomar una foto con la cámara del computador (la celda pide permiso al navegador) o subir una imagen.

In [ ]:
def foto_camara():
    """Toma una foto con la cámara del navegador (solo en Colab)."""
    import base64, io
    from PIL import Image
    from IPython.display import display, Javascript
    from google.colab.output import eval_js
    display(Javascript("""
      async function foto() {
        const video = document.createElement('video'); const boton = document.createElement('button');
        boton.textContent = '📸 Tomar foto'; document.body.appendChild(video); document.body.appendChild(boton);
        const flujo = await navigator.mediaDevices.getUserMedia({video: true});
        video.srcObject = flujo; await video.play();
        await new Promise(r => boton.onclick = r);
        const lienzo = document.createElement('canvas');
        lienzo.width = video.videoWidth; lienzo.height = video.videoHeight; lienzo.getContext('2d').drawImage(video, 0, 0);
        flujo.getTracks().forEach(t => t.stop()); video.remove(); boton.remove();
        return lienzo.toDataURL('image/jpeg', 0.9);
      }"""))
    datos = eval_js("foto()")
    return Image.open(io.BytesIO(base64.b64decode(datos.split(",")[1]))).convert("RGB")

if EN_COLAB and not PRUEBA_RAPIDA:
    print("Para usarla: mi_foto = foto_camara(), y después clasificador(mi_foto) o la descripción con BLIP.")
else:
    print("La cámara solo funciona dentro de Colab.")

## Pruébalo tú

Genera una imagen de algo que **no** esté en las 1.000 categorías de ImageNet (por ejemplo, «una empanada chilena») y
guarda en `etiqueta_vit` lo que ViT cree que es. ¿Qué te dice eso sobre los clasificadores de categorías fijas?
¿Le va mejor a CLIP con frases tuyas?

In [ ]:
etiqueta_vit = None   # ← tu código

In [ ]:
# Comprobación
if etiqueta_vit is None:
    print("Aún no lo resuelves.")
elif isinstance(etiqueta_vit, str):
    print("✅ Registrado: compáralo con lo que responde CLIP")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- **Difusión:** de ruido a imagen, guiada por un texto; la semilla la hace repetible.
- **ViT** clasifica en categorías fijas; **BLIP** describe con palabras; **CLIP** compara con frases libres.
- Usa estos modelos de forma responsable: revisa su licencia y nunca generes imágenes que suplanten a personas.

**Fuentes:** [Diffusers · Stable Diffusion](https://huggingface.co/docs/diffusers/using-diffusers/conditional_image_generation) · [Stable Diffusion 1.5 y su licencia](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) · [ViT-base](https://huggingface.co/google/vit-base-patch16-224) · [BLIP-base](https://huggingface.co/Salesforce/blip-image-captioning-base) · [CLIP (OpenAI)](https://github.com/openai/CLIP) · [FAQ de Colab · usos prohibidos](https://research.google.com/colaboratory/faq.html)